# Comparison of log p(x) estimators in VAEs

In this notebook, we will explore four different ways to estimate the marginal likelihood $\log p(x)$ of our data given a pre-trained VAE model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from vae import *
import numpy as np
import math
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

The exact formula for the marginal likelihood is given by:
$$
\log p(x) = \log \int p(x|z)p(z) dz
$$

However, this integral is often intractable, as it requires summing over all possible latent variable configurations. Therefore, we will explore four different estimators for $\log p(x)$.

### 1. Sampling $z$ in $R^d$ (Uniform Space)

Our first approach is to uniformly sample points in the latent space $R^d$ and evaluate the integrand $p(x|z)p(z)$ at those points. This method is straightforward, but it can be very inefficient since probabilities are often concentrated in a small region of the latent space, and uniform sampling may miss these regions.

In [ ]:
def estimate_log_px_uniform(vae, x, num_samples=1000, bounds=(-5, 5)):
    # x shape: [1, input_dim]
    latent_dim = vae.encoder.latent_dim
    # Uniform sampling of z in R^d within bounds
    z = torch.empty(num_samples, latent_dim).uniform_(bounds[0], bounds[1]).to(device)
    
    # Reconstruction and evaluation
    x_recon = vae.decoder(z)
    log_p_x_z = -F.binary_cross_entropy(x_recon, x.expand(num_samples, -1), reduction='none').sum(dim=1)
    
    # Log p(z) assuming N(0, I)
    log_p_z = -0.5 * torch.sum(z**2 + math.log(2 * math.pi), dim=1)
    
    # Volume of the sampled space
    vol = (bounds[1] - bounds[0]) ** latent_dim
    
    # Monte Carlo approximation
    log_weights = log_p_x_z + log_p_z + math.log(vol)
    log_px = torch.logsumexp(log_weights, dim=0) - math.log(num_samples)
    return log_px.item()

### 2. Sampling z ~ p(z)
A slightly better alternative is to sample directly from our prior distribution $p(z) = \mathcal{N}(0, I)$.

In [ ]:
def estimate_log_px_prior(vae, x, num_samples=1000):
    latent_dim = vae.encoder.latent_dim
    # z ~ N(0, I)
    z = torch.randn(num_samples, latent_dim).to(device)
    
    x_recon = vae.decoder(z)
    log_p_x_z = -F.binary_cross_entropy(x_recon, x.expand(num_samples, -1), reduction='none').sum(dim=1)
    
    # log p(x) ≈ log( 1/K sum(p(x|z)) )
    log_px = torch.logsumexp(log_p_x_z, dim=0) - math.log(num_samples)
    return log_px.item()

### 3. Encoding according to q(z|x) (Importance Sampling)
By using the encoder to guide the sampling, we obtain an unbiased estimator with much lower variance: the IWAE (Importance Weighted Autoencoder) bound.

In [ ]:
def estimate_log_px_importance_sampling(vae, x, num_samples=1000):
    # Get parameters of q(z|x)
    mu, log_var = vae.encoder(x)
    std = torch.exp(0.5 * log_var)
    
    # Sample z ~ q(z|x)
    z = mu + std * torch.randn(num_samples, vae.encoder.latent_dim).to(device)
    
    # Evaluate log p(x|z)
    x_recon = vae.decoder(z)
    log_p_x_z = -F.binary_cross_entropy(x_recon, x.expand(num_samples, -1), reduction='none').sum(dim=1)
    
    # Evaluate log p(z)
    log_p_z = -0.5 * torch.sum(z**2 + math.log(2 * math.pi), dim=1)
    
    # Evaluate log q(z|x)
    log_q_z_x = -0.5 * torch.sum(((z - mu) / std)**2 + log_var + math.log(2 * math.pi), dim=1)
    
    # log p(x) ≈ log( 1/K sum( p(x|z)p(z) / q(z|x) ) )
    log_weights = log_p_x_z + log_p_z - log_q_z_x
    log_px = torch.logsumexp(log_weights, dim=0) - math.log(num_samples)
    return log_px.item()

### 4. Estimating with the ELBO (Evidence Lower Bound)
The ELBO is a strict lower bound of $\log p(x)$. It is the standard optimization method in a VAE.

In [ ]:
def estimate_elbo(vae, x):
    mu, log_var = vae.encoder(x)
    z = vae.reparameterize(mu, log_var)
    x_recon = vae.decoder(z)
    
    # Reconstruction Term: E_q[log p(x|z)]
    recon_loss = -F.binary_cross_entropy(x_recon, x, reduction='sum')
    
    # KL Divergence: D_KL(q(z|x) || p(z))
    kl_divergence = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    
    # ELBO = Reconstruction - KL
    elbo = recon_loss - kl_divergence
    return elbo.item()